In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.rcParams['font.family'] ='Malgun Gothic'
plt.rcParams['axes.unicode_minus'] =False

In [ ]:
import pandas as pd

df1 = pd.read_csv('factory_hourly_final.csv')
df = df1.copy()

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

# 1. K값을 1부터 10까지 바꿔가며 SSE 계산
sse = []
k_range = range(1, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42)
    kmeans.fit(scaled_data)
    sse.append(kmeans.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(k_range, sse, marker='o', linestyle='--', color='b')
plt.title('Elbow Method for Optimal K')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('SSE (Inertia)')
plt.xticks(k_range)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

features_to_use = [
    'activePower', 'total_pf',
    'tw_avg_pf_R', 'tw_avg_pf_S', 'tw_avg_pf_T', 'tw_avg_reactivePow',
    'current_CV', 'current_unbal_ratio', 'current_change_acc',
    'voltage_CV_RS', 'voltage_CV_ST', 'voltage_CV_TR'
]

module_stats = df.groupby('module(equipment)')[features_to_use].mean().fillna(0)
scaler = StandardScaler()
scaled_data = scaler.fit_transform(module_stats)

n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, init='k-means++', random_state=42)
module_stats['cluster'] = kmeans.fit_predict(scaled_data)

pca = PCA(n_components=2)
pca_transformed = pca.fit_transform(scaled_data)
module_stats['pca_x'] = pca_transformed[:, 0]
module_stats['pca_y'] = pca_transformed[:, 1]

plt.figure(figsize=(12, 8))
sns.scatterplot(x='pca_x', y='pca_y', hue='cluster', size='activePower', data=module_stats, palette='Set1', sizes=(50, 500), alpha=0.7)
for i in range(module_stats.shape[0]):
    plt.text(module_stats.pca_x[i]+0.02, module_stats.pca_y[i]+0.02, module_stats.index[i], fontsize=9)
plt.title('Equipment Clustering based on Power Quality & Stability')
plt.grid(True, alpha=0.3)
plt.show()

cluster_report = module_stats.groupby('cluster').mean()
print(cluster_report)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

def get_normalized_pattern(df, column_name):
    pivot = df.pivot_table(index='module(equipment)', columns='hour', values=column_name, aggfunc='mean').fillna(0)
    scaler = MinMaxScaler()
    return pd.DataFrame(scaler.fit_transform(pivot.T).T, index=pivot.index, columns=[f'{column_name}_h{i}' for i in range(24)])

active_p = get_normalized_pattern(df, 'activePower')
reactive_p = get_normalized_pattern(df, 'tw_avg_reactivePow')
hourly_p = get_normalized_pattern(df, 'hourly_pow')

quality_features = ['total_pf', 'current_CV', 'current_unbal_ratio', 'current_change_acc',
                    'tw_avg_pf_R', 'tw_avg_pf_S', 'tw_avg_pf_T', 'tw_avg_reactivePow',
                    'voltage_CV_RS', 'voltage_CV_ST', 'voltage_CV_TR']
module_quality = df.groupby('module(equipment)')[quality_features].mean().fillna(0)
final_combined = pd.concat([module_quality, active_p, reactive_p, hourly_p], axis=1)

scaler_final = StandardScaler()
final_scaled = scaler_final.fit_transform(final_combined)
kmeans = KMeans(n_clusters=4, init='k-means++', random_state=42)
final_combined['cluster'] = kmeans.fit_predict(final_scaled)

fig, axes = plt.subplots(4, 3, figsize=(16, 20))
for i in range(4):
    cluster_idx = final_combined['cluster'] == i
    axes[i, 0].plot(active_p[cluster_idx].T, color='gray', alpha=0.2)
    axes[i, 0].plot(active_p[cluster_idx].mean(), color='blue', linewidth=3)
    axes[i, 1].plot(reactive_p[cluster_idx].T, color='gray', alpha=0.2)
    axes[i, 1].plot(reactive_p[cluster_idx].mean(), color='red', linewidth=3)
    axes[i, 2].plot(hourly_p[cluster_idx].T, color='gray', alpha=0.2)
    axes[i, 2].plot(hourly_p[cluster_idx].mean(), color='green', linewidth=3)
plt.tight_layout()
plt.show()

Cluster (0)
- 2호기
- 우측분전반2

Cluster (1)
- 3호기
- 예비건조기
- 6호기
- 분쇄기1
- 좌측분전반

Cluster (2)
- PM-3
- 4호기
- 호이스트
- 분쇄기2

Cluster (3)
- 우측분전반1
- L-1 전등

Cluster 0: 주간 무효전력이 높고 기저부하 성격.
Cluster 1: 유효/무효전력이 동기화되며 변동성이 큰 작업 기반 부하.
Cluster 2: 자정 부근 기동 부하가 크고 이후 빠르게 안정되는 패턴.
Cluster 3: 사용량이 낮고 외부 환경/사용자 개입 영향이 큰 패턴.